In [1]:
import matplotlib
print(matplotlib.__version__)


3.10.0


In [3]:
import pandas as pd
from datetime import datetime, timedelta

# Path to your CSV file
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"

# Read CSV file
df = pd.read_csv(file_path)

# Starting time and interval
start_time = datetime(2025, 10, 4, 0, 0, 0)  # 04/10/2025 12:00 AM
time_interval = timedelta(minutes=30)

# Generate new time column
df['Time'] = [start_time + i * time_interval for i in range(len(df))]

# Format time as "dd/mm/yyyy hh:mm:ss AM/PM"
df['Time'] = df['Time'].dt.strftime("%d/%m/%Y %I:%M:%S %p")

# Save back to same CSV (or you can give a new name if you want backup)
df.to_csv(file_path, index=False)

print("✅ Time column updated successfully and file saved at:")
print(file_path)


✅ Time column updated successfully and file saved at:
D:\ChipVista\Projects\decaying fruits\DATALOG.csv


In [7]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"

# --- Load dataset ---
df = pd.read_csv(file_path)

# Keep only needed columns
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M')

# --- Add hour & day info ---
df['Hour'] = df['Time'].dt.hour
df['Day'] = df['Time'].dt.date

# --- Step 1: Calculate hourly mean for each day ---
hourly_data = df.groupby(['Day', 'Hour'])[['mq2', 'mq4']].mean().reset_index()

# --- Step 2: Plot graphs per day ---
unique_days = hourly_data['Day'].unique()

for i, day in enumerate(unique_days, start=1):
    subset = hourly_data[hourly_data['Day'] == day]
    
    # MQ2
    plt.figure(figsize=(12, 5))
    plt.plot(subset['Hour'], subset['mq2'], marker='o', color='blue')
    plt.title(f"MQ2 Values - Day {i} ({day})")
    plt.xlabel("Hour of Day (0–23)")
    plt.ylabel("MQ2 Value")
    plt.xticks(range(0, 24))
    plt.grid(True)
    plt.savefig(os.path.join(save_folder, f"Day{i}_MQ2.png"))
    plt.close()

    # MQ4
    plt.figure(figsize=(12, 5))
    plt.plot(subset['Hour'], subset['mq4'], marker='o', color='green')
    plt.title(f"MQ4 Values - Day {i} ({day})")
    plt.xlabel("Hour of Day (0–23)")
    plt.ylabel("MQ4 Value")
    plt.xticks(range(0, 24))
    plt.grid(True)
    plt.savefig(os.path.join(save_folder, f"Day{i}_MQ4.png"))
    plt.close()

    # Merged (MQ2 + MQ4)
    plt.figure(figsize=(12, 5))
    plt.plot(subset['Hour'], subset['mq2'], marker='o', label='MQ2', color='blue')
    plt.plot(subset['Hour'], subset['mq4'], marker='o', label='MQ4', color='green')
    plt.title(f"MQ2 & MQ4 Values - Day {i} ({day})")
    plt.xlabel("Hour of Day (0–23)")
    plt.ylabel("Sensor Values")
    plt.xticks(range(0, 24))
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(save_folder, f"Day{i}_Merged.png"))
    plt.close()

# --- Step 3: Combined overall timeline ---
plt.figure(figsize=(14, 6))
plt.plot(df['Time'], df['mq2'], label='MQ2', color='blue')
plt.plot(df['Time'], df['mq4'], label='MQ4', color='green')
plt.title("Overall MQ2 & MQ4 Values Across All Days")
plt.xlabel("Time")
plt.ylabel("Sensor Values")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(save_folder, "Overall_MQ2_MQ4.png"))
plt.close()

print("✅ Graphs generated successfully in:")
print(save_folder)


✅ Graphs generated successfully in:
D:\ChipVista\Projects\decaying fruits


In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"

# --- Load dataset ---
df = pd.read_csv(file_path)

# Keep only required columns
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M')

# --- Add day and half-hour index ---
df['Day'] = df['Time'].dt.date

# Since readings are every 30 minutes, create a reading index (0–47 for each day)
df['HalfHourIndex'] = df.groupby('Day').cumcount()

# --- Step 1: Plot graphs per day ---
unique_days = df['Day'].unique()

for i, day in enumerate(unique_days, start=1):
    subset = df[df['Day'] == day].reset_index(drop=True)

    # X-axis setup
    hour_ticks = np.arange(0, 48, 2)         # every 1 hour = 2 readings
    minor_ticks = np.arange(1, 48, 2)        # 30-minute marks
    hour_labels = np.arange(0, 24, 1)        # 0–23 hours (matches 24 tick points)

    # --- MQ2 Plot ---
    plt.figure(figsize=(12, 5))
    plt.plot(subset['HalfHourIndex'], subset['mq2'], marker='o', color='blue')
    plt.title(f"MQ2 Values - Day {i} ({day})\n(Readings taken every 30 minutes)")
    plt.xlabel("Time (Hours)")
    plt.ylabel("MQ2 Value")

    plt.xticks(hour_ticks, hour_labels)
    plt.tick_params(axis='x', which='major', length=8, width=1.2)
    plt.tick_params(axis='x', which='minor', length=4, color='gray')
    plt.gca().set_xticks(minor_ticks, minor=True)
    plt.grid(True, which='major', linestyle='-', linewidth=0.6)
    plt.grid(True, which='minor', linestyle=':', linewidth=0.4)

    plt.savefig(os.path.join(save_folder, f"Day{i}_MQ2.png"))
    plt.close()

    # --- MQ4 Plot ---
    plt.figure(figsize=(12, 5))
    plt.plot(subset['HalfHourIndex'], subset['mq4'], marker='o', color='green')
    plt.title(f"MQ4 Values - Day {i} ({day})\n(Readings taken every 30 minutes)")
    plt.xlabel("Time (Hours)")
    plt.ylabel("MQ4 Value")

    plt.xticks(hour_ticks, hour_labels)
    plt.tick_params(axis='x', which='major', length=8, width=1.2)
    plt.tick_params(axis='x', which='minor', length=4, color='gray')
    plt.gca().set_xticks(minor_ticks, minor=True)
    plt.grid(True, which='major', linestyle='-', linewidth=0.6)
    plt.grid(True, which='minor', linestyle=':', linewidth=0.4)

    plt.savefig(os.path.join(save_folder, f"Day{i}_MQ4.png"))
    plt.close()

    # --- Combined MQ2 + MQ4 Plot ---
    plt.figure(figsize=(12, 5))
    plt.plot(subset['HalfHourIndex'], subset['mq2'], marker='o', label='MQ2', color='blue')
    plt.plot(subset['HalfHourIndex'], subset['mq4'], marker='o', label='MQ4', color='green')
    plt.title(f"MQ2 & MQ4 Values - Day {i} ({day})\n(Readings taken every 30 minutes)")
    plt.xlabel("Time (Hours)")
    plt.ylabel("Sensor Values")
    plt.legend()

    plt.xticks(hour_ticks, hour_labels)
    plt.tick_params(axis='x', which='major', length=8, width=1.2)
    plt.tick_params(axis='x', which='minor', length=4, color='gray')
    plt.gca().set_xticks(minor_ticks, minor=True)
    plt.grid(True, which='major', linestyle='-', linewidth=0.6)
    plt.grid(True, which='minor', linestyle=':', linewidth=0.4)

    plt.savefig(os.path.join(save_folder, f"Day{i}_Merged.png"))
    plt.close()

# --- Step 2: Overall timeline plot ---
plt.figure(figsize=(14, 6))
plt.plot(df['Time'], df['mq2'], label='MQ2', color='blue')
plt.plot(df['Time'], df['mq4'], label='MQ4', color='green')
plt.title("Overall MQ2 & MQ4 Values Across All Days\n(Readings taken every 30 minutes)")
plt.xlabel("Time")
plt.ylabel("Sensor Values")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(save_folder, "Overall_MQ2_MQ4.png"))
plt.close()

print("✅ Graphs generated successfully in:")
print(save_folder)


✅ Graphs generated successfully in:
D:\ChipVista\Projects\decaying fruits


In [13]:
# with explaination 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"

# --- Load dataset ---
df = pd.read_csv(file_path)
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M')
df['Day'] = df['Time'].dt.date
df['HalfHourIndex'] = df.groupby('Day').cumcount()

# --- Explanation text (to print under each graph) ---
explanation_text = (
    "Explanation:\n"
    "• Y-axis shows the ADC value (0–1023) from the gas sensor.\n"
    "• Each ADC value is the average of 8 readings using:  acc += analogRead(pin);  delay(2);\n"
    "  → ADCavg = (acc / 8)\n"
    "• ADC represents the digital conversion of the analog sensor voltage:\n"
    "      Voltage = (ADC / 1023) × 5.0\n"
    "• Higher ADC → higher sensor voltage → higher gas concentration.\n"
)

# --- Create graphs for each day ---
unique_days = df['Day'].unique()

for i, day in enumerate(unique_days, start=1):
    subset = df[df['Day'] == day].reset_index(drop=True)

    hour_ticks = np.arange(0, 48, 2)
    minor_ticks = np.arange(1, 48, 2)
    hour_labels = np.arange(0, 24, 1)

    # === Function to add graph + explanation ===
    def make_graph(sensor_name, color, y_values, filename, label_text):
        fig, ax = plt.subplots(figsize=(12, 8))  # Taller figure for space below
        plt.subplots_adjust(bottom=0.35)  # leave room for explanation text

        ax.plot(subset['HalfHourIndex'], y_values, marker='o', color=color)
        ax.set_title(f"{sensor_name} Values - Day {i} ({day})\n(Readings every 30 minutes)")
        ax.set_xlabel("Time (Hours)")
        ax.set_ylabel(f"{sensor_name} ADC Value")

        ax.set_xticks(hour_ticks)
        ax.set_xticklabels(hour_labels)
        ax.tick_params(axis='x', which='major', length=8, width=1.2)
        ax.tick_params(axis='x', which='minor', length=4, color='gray')
        ax.set_xticks(minor_ticks, minor=True)
        ax.grid(True, which='major', linestyle='-', linewidth=0.6)
        ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

        # Add text below plot
        plt.figtext(
            0.05, 0.02, explanation_text, wrap=True, ha="left", va="bottom", fontsize=10,
            bbox=dict(facecolor='whitesmoke', alpha=0.6, boxstyle="round,pad=0.5")
        )

        plt.savefig(os.path.join(save_folder, filename))
        plt.close()

    # --- MQ2 Plot ---
    make_graph("MQ2", "blue", subset['mq2'], f"Day{i}_MQ2.png", "MQ2")

    # --- MQ4 Plot ---
    make_graph("MQ4", "green", subset['mq4'], f"Day{i}_MQ4.png", "MQ4")

    # --- Combined Plot ---
    fig, ax = plt.subplots(figsize=(12, 8))
    plt.subplots_adjust(bottom=0.35)
    ax.plot(subset['HalfHourIndex'], subset['mq2'], marker='o', label='MQ2', color='blue')
    ax.plot(subset['HalfHourIndex'], subset['mq4'], marker='o', label='MQ4', color='green')
    ax.set_title(f"MQ2 & MQ4 Values - Day {i} ({day})\n(Readings every 30 minutes)")
    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel("Sensor ADC Values")
    ax.legend()

    ax.set_xticks(hour_ticks)
    ax.set_xticklabels(hour_labels)
    ax.tick_params(axis='x', which='major', length=8, width=1.2)
    ax.tick_params(axis='x', which='minor', length=4, color='gray')
    ax.set_xticks(minor_ticks, minor=True)
    ax.grid(True, which='major', linestyle='-', linewidth=0.6)
    ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

    plt.figtext(
        0.05, 0.02, explanation_text, wrap=True, ha="left", va="bottom", fontsize=10,
        bbox=dict(facecolor='whitesmoke', alpha=0.6, boxstyle="round,pad=0.5")
    )

    plt.savefig(os.path.join(save_folder, f"Day{i}_Merged.png"))
    plt.close()

# --- Step 2: Overall Timeline Plot ---
fig, ax = plt.subplots(figsize=(14, 8))
plt.subplots_adjust(bottom=0.35)
ax.plot(df['Time'], df['mq2'], label='MQ2', color='blue')
ax.plot(df['Time'], df['mq4'], label='MQ4', color='green')
ax.set_title("Overall MQ2 & MQ4 Values Across All Days\n(Readings every 30 minutes)")
ax.set_xlabel("Time")
ax.set_ylabel("Sensor ADC Values")
ax.legend()
ax.grid(True)
plt.tight_layout(rect=[0, 0.2, 1, 1])

plt.figtext(
    0.05, 0.02, explanation_text, wrap=True, ha="left", va="bottom", fontsize=10,
    bbox=dict(facecolor='whitesmoke', alpha=0.6, boxstyle="round,pad=0.5")
)

plt.savefig(os.path.join(save_folder, "Overall_MQ2_MQ4.png"))
plt.close()

print("✅ Graphs with explanations generated successfully in:")
print(save_folder)


✅ Graphs with explanations generated successfully in:
D:\ChipVista\Projects\decaying fruits


In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import date, timedelta

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"

# --- Load dataset ---
df = pd.read_csv(file_path)
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M', errors='coerce')

# --- Add day and half-hour index ---
df['Day'] = df['Time'].dt.date
df['HalfHourIndex'] = df.groupby('Day').cumcount()

# --- Generate artificial day sequence starting from 2025-10-03 ---
unique_days = df['Day'].unique()
base_date = date(2025, 10, 3)
display_days = [base_date + timedelta(days=i) for i in range(len(unique_days))]

# --- Explanation text for graphs ---
explanation_text = (
    "Explanation:\n"
    "• Y-axis shows the ADC value (0–1023) from the MQ sensor module.\n"
    "• Each ADC value is the average of 8 samples → ADCavg = (sum of 8 readings) / 8.\n"
    "• Formula: Voltage = (ADC / 1023) × 5.0 converts ADC to sensor voltage.\n"
    "• Higher ADC = higher sensor voltage = higher gas concentration detected.\n"
)

# --- Create graphs for each day ---
for i, (real_day, display_day) in enumerate(zip(unique_days, display_days), start=1):
    subset = df[df['Day'] == real_day].reset_index(drop=True)

    hour_ticks = np.arange(0, 48, 2)
    minor_ticks = np.arange(1, 48, 2)
    hour_labels = np.arange(0, 24, 1)

    def make_graph(sensor_name, color, y_values, filename):
        fig, ax = plt.subplots(figsize=(12, 9))
        plt.subplots_adjust(bottom=0.45)  # more space for explanation box

        ax.plot(subset['HalfHourIndex'], y_values, marker='o', color=color)
        ax.set_title(f"{sensor_name} Values - Day {i} ({display_day})\n(Readings every 30 minutes)")
        ax.set_xlabel("Time (Hours)")
        ax.set_ylabel(f"{sensor_name} ADC Value")

        ax.set_xticks(hour_ticks)
        ax.set_xticklabels(hour_labels)
        ax.tick_params(axis='x', which='major', length=8, width=1.2)
        ax.tick_params(axis='x', which='minor', length=4, color='gray')
        ax.set_xticks(minor_ticks, minor=True)
        ax.grid(True, which='major', linestyle='-', linewidth=0.6)
        ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

        plt.figtext(
            0.05, 0.05,
            explanation_text,
            wrap=True,
            ha="left",
            va="bottom",
            fontsize=10,
            bbox=dict(facecolor='whitesmoke', alpha=0.7, boxstyle="round,pad=0.5")
        )

        plt.savefig(os.path.join(save_folder, filename))
        plt.close()

    # MQ2 plot
    make_graph("MQ2", "blue", subset['mq2'], f"Day{i}_MQ2.png")

    # MQ4 plot
    make_graph("MQ4", "green", subset['mq4'], f"Day{i}_MQ4.png")

    # Combined plot
    fig, ax = plt.subplots(figsize=(12, 9))
    plt.subplots_adjust(bottom=0.45)
    ax.plot(subset['HalfHourIndex'], subset['mq2'], marker='o', label='MQ2', color='blue')
    ax.plot(subset['HalfHourIndex'], subset['mq4'], marker='o', label='MQ4', color='green')
    ax.set_title(f"MQ2 & MQ4 Values - Day {i} ({display_day})\n(Readings every 30 minutes)")
    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel("Sensor ADC Values")
    ax.legend()

    ax.set_xticks(hour_ticks)
    ax.set_xticklabels(hour_labels)
    ax.tick_params(axis='x', which='major', length=8, width=1.2)
    ax.tick_params(axis='x', which='minor', length=4, color='gray')
    ax.set_xticks(minor_ticks, minor=True)
    ax.grid(True, which='major', linestyle='-', linewidth=0.6)
    ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

    plt.figtext(
        0.05, 0.05,
        explanation_text,
        wrap=True,
        ha="left",
        va="bottom",
        fontsize=10,
        bbox=dict(facecolor='whitesmoke', alpha=0.7, boxstyle="round,pad=0.5")
    )

    plt.savefig(os.path.join(save_folder, f"Day{i}_Merged.png"))
    plt.close()

# --- Overall timeline plot ---
fig, ax = plt.subplots(figsize=(14, 9))
plt.subplots_adjust(bottom=0.45)
ax.plot(df['Time'], df['mq2'], label='MQ2', color='blue')
ax.plot(df['Time'], df['mq4'], label='MQ4', color='green')
ax.set_title("Overall MQ2 & MQ4 Values Across All Days\n(Readings every 30 minutes)")
ax.set_xlabel("Time")
ax.set_ylabel("Sensor ADC Values")
ax.legend()
ax.grid(True)
plt.tight_layout(rect=[0, 0.25, 1, 1])

plt.figtext(
    0.05, 0.05,
    explanation_text,
    wrap=True,
    ha="left",
    va="bottom",
    fontsize=10,
    bbox=dict(facecolor='whitesmoke', alpha=0.7, boxstyle="round,pad=0.5")
)

plt.savefig(os.path.join(save_folder, "Overall_MQ2_MQ4.png"))
plt.close()

print("✅ Graphs generated successfully with starting date labels from 2025-10-03!")


✅ Graphs generated successfully with starting date labels from 2025-10-03!


In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import date, timedelta

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"

# --- Load dataset ---
df = pd.read_csv(file_path)
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M', errors='coerce')

# --- Add day and half-hour index ---
df['Day'] = df['Time'].dt.date
df['HalfHourIndex'] = df.groupby('Day').cumcount()

# --- Generate artificial day sequence starting from 2025-10-03 ---
unique_days = df['Day'].unique()
base_date = date(2025, 10, 3)
display_days = [base_date + timedelta(days=i) for i in range(len(unique_days))]

# --- Explanation text for graphs ---
explanation_text = (
    "Explanation:\n"
    "• Y-axis shows the ADC value (0–1023) measured by each MQ sensor module.\n"
    "• Each ADC value = average of 8 samples → ADCavg = (sum of 8 readings) / 8.\n"
    "• Conversion: Voltage = (ADC / 1023) × 5.0 gives sensor output voltage.\n"
    "• Higher ADC → higher voltage → higher gas concentration detected.\n"
)

# --- Create graphs for each day ---
for i, (real_day, display_day) in enumerate(zip(unique_days, display_days), start=1):
    subset = df[df['Day'] == real_day].reset_index(drop=True)

    hour_ticks = np.arange(0, 48, 2)
    minor_ticks = np.arange(1, 48, 2)
    hour_labels = np.arange(0, 24, 1)

    def make_graph(sensor_name, color, y_values, filename):
        fig, ax = plt.subplots(figsize=(12, 10))
        plt.subplots_adjust(bottom=0.35)  # increased bottom space for larger text area

        ax.plot(subset['HalfHourIndex'], y_values, marker='o', color=color)
        ax.set_title(f"{sensor_name} Values - Day {i} ({display_day})\n(Readings every 30 minutes)")
        ax.set_xlabel("Time (Hours)")
        ax.set_ylabel(f"{sensor_name} ADC Value")

        ax.set_xticks(hour_ticks)
        ax.set_xticklabels(hour_labels)
        ax.tick_params(axis='x', which='major', length=8, width=1.2)
        ax.tick_params(axis='x', which='minor', length=4, color='gray')
        ax.set_xticks(minor_ticks, minor=True)
        ax.grid(True, which='major', linestyle='-', linewidth=0.6)
        ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

        # Enlarged explanation text box
        plt.figtext(
            0.05, 0.02,
            explanation_text,
            wrap=True,
            ha="left",
            va="bottom",
            fontsize=11,
            bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=0.8")
        )

        plt.savefig(os.path.join(save_folder, filename), dpi=300)
        plt.close()

    # MQ2 plot
    make_graph("MQ2", "blue", subset['mq2'], f"Day{i}_MQ2.png")

    # MQ4 plot
    make_graph("MQ4", "green", subset['mq4'], f"Day{i}_MQ4.png")

    # Combined plot (both sensors)
    fig, ax = plt.subplots(figsize=(12, 10))
    plt.subplots_adjust(bottom=0.3)

    ax.plot(subset['HalfHourIndex'], subset['mq2'], marker='o', label='MQ2', color='blue')
    ax.plot(subset['HalfHourIndex'], subset['mq4'], marker='o', label='MQ4', color='green')

    ax.set_title(f"MQ2 & MQ4 Values - Day {i} ({display_day})\n(Readings every 30 minutes)")
    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel("Sensor ADC Values")
    ax.legend()

    ax.set_xticks(hour_ticks)
    ax.set_xticklabels(hour_labels)
    ax.tick_params(axis='x', which='major', length=8, width=1.2)
    ax.tick_params(axis='x', which='minor', length=4, color='gray')
    ax.set_xticks(minor_ticks, minor=True)
    ax.grid(True, which='major', linestyle='-', linewidth=0.6)
    ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

    plt.figtext(
        0.06, 0.03,
        explanation_text,
        wrap=True,
        ha="left",
        va="bottom",
        fontsize=11,
        bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=0.8")
    )

    plt.savefig(os.path.join(save_folder, f"Day{i}_Merged.png"), dpi=300)
    plt.close()

# --- Overall timeline plot ---
fig, ax = plt.subplots(figsize=(14, 10))
plt.subplots_adjust(bottom=0.3)
ax.plot(df['Time'], df['mq2'], label='MQ2', color='blue')
ax.plot(df['Time'], df['mq4'], label='MQ4', color='green')
ax.set_title("Overall MQ2 & MQ4 Values Across All Days\n(Readings every 30 minutes)")
ax.set_xlabel("Time")
ax.set_ylabel("Sensor ADC Values")
ax.legend()
ax.grid(True)
plt.tight_layout(rect=[0, 0.3, 1, 1])

plt.figtext(
    0.05, 0.02,
    explanation_text,
    wrap=True,
    ha="left",
    va="bottom",
    fontsize=11,
    bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=0.8")
)

plt.savefig(os.path.join(save_folder, "Overall_MQ2_MQ4.png"), dpi=300)
plt.close()

print("✅ Graphs generated successfully")


✅ Graphs generated successfully with larger explanation box and start date 2025-10-03!


In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import date, timedelta

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"
os.makedirs(save_folder, exist_ok=True)

# --- Load dataset ---
df = pd.read_csv(file_path)
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime (your format) ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M', errors='coerce')

# --- Add day and half-hour index ---
df['Day'] = df['Time'].dt.date
df['HalfHourIndex'] = df.groupby('Day').cumcount()

# --- Generate artificial day sequence starting from 2025-10-03 ---
unique_days = df['Day'].unique()
base_date = date(2025, 10, 3)
display_days = [base_date + timedelta(days=i) for i in range(len(unique_days))]

# --- SUPER SIMPLE explanation for PPM graphs ---
explanation_text = (
    "• Y-axis shows gas concentration in ppm (parts per million), higher = more target gas.\n"
    "• Each point comes from: ADC → Vout → Rs = RL·(5−Vout)/Vout → (Rs/R0) → ppm conversion.\n"
    "• Rs = sensor resistance (changes with gas), R0 = resistance in clean air, RL = load resistor.\n"
    "• Rs/R0 shows overall gas effect (mixture of gases) — ppm indicates specific target gas level.\n"
    "• MQ-2 ppm ≈ 50·(Rs/R0)^−2.3 for hydrocarbons like ethylene, propane, or smoke.\n"
    "• MQ-4 ppm ≈ 1000·(Rs/R0)^−2.95 for methane (CH₄) concentration.\n"
    "• As gas increases, Rs/R0 decreases → ppm increases. Peaks = higher gas; flat = clean air."
)


# --- Create graphs for each day ---
for i, (real_day, display_day) in enumerate(zip(unique_days, display_days), start=1):
    subset = df[df['Day'] == real_day].reset_index(drop=True)

    hour_ticks = np.arange(0, 48, 2)
    minor_ticks = np.arange(1, 48, 2)
    hour_labels = np.arange(0, 24, 1)

    def make_graph(sensor_name, color, y_values, filename):
        fig, ax = plt.subplots(figsize=(12, 10))
        plt.subplots_adjust(bottom=0.35)

        ax.plot(subset['HalfHourIndex'], y_values, marker='o', color=color)
        ax.set_title(f"{sensor_name} (ppm) - Day {i} ({display_day})\n(Readings every 30 minutes)")
        ax.set_xlabel("Time (Hours)")
        ax.set_ylabel(f"{sensor_name} Concentration (ppm)")

        ax.set_xticks(hour_ticks)
        ax.set_xticklabels(hour_labels)
        ax.tick_params(axis='x', which='major', length=8, width=1.2)
        ax.tick_params(axis='x', which='minor', length=4, color='gray')
        ax.set_xticks(minor_ticks, minor=True)
        ax.grid(True, which='major', linestyle='-', linewidth=0.6)
        ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

        # Explanation box
        plt.figtext(
            0.05, 0.02,
            explanation_text,
            wrap=True,
            ha="left",
            va="bottom",
            fontsize=11,
            bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=0.8")
        )

        plt.savefig(os.path.join(save_folder, filename), dpi=300)
        plt.close()

    # MQ2 plot
    make_graph("MQ2", "blue", subset['mq2'], f"Day{i}_MQ2_ppm.png")

    # MQ4 plot
    make_graph("MQ4", "green", subset['mq4'], f"Day{i}_MQ4_ppm.png")

    # Combined plot (both sensors)
    fig, ax = plt.subplots(figsize=(12, 10))
    plt.subplots_adjust(bottom=0.3)

    ax.plot(subset['HalfHourIndex'], subset['mq2'], marker='o', label='MQ2 (ppm)', color='blue')
    ax.plot(subset['HalfHourIndex'], subset['mq4'], marker='o', label='MQ4 (ppm)', color='green')

    ax.set_title(f"MQ2 & MQ4 (ppm) - Day {i} ({display_day})\n(Readings every 30 minutes)")
    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel("Concentration (ppm)")
    ax.legend()

    ax.set_xticks(hour_ticks)
    ax.set_xticklabels(hour_labels)
    ax.tick_params(axis='x', which='major', length=8, width=1.2)
    ax.tick_params(axis='x', which='minor', length=4, color='gray')
    ax.set_xticks(minor_ticks, minor=True)
    ax.grid(True, which='major', linestyle='-', linewidth=0.6)
    ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

    plt.figtext(
        0.06, 0.03,
        explanation_text,
        wrap=True,
        ha="left",
        va="bottom",
        fontsize=11,
        bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=0.8")
    )

    plt.savefig(os.path.join(save_folder, f"Day{i}_Merged_ppm.png"), dpi=300)
    plt.close()

# --- Overall timeline plot ---
fig, ax = plt.subplots(figsize=(14, 10))
plt.subplots_adjust(bottom=0.3)
ax.plot(df['Time'], df['mq2'], label='MQ2 (ppm)', color='blue')
ax.plot(df['Time'], df['mq4'], label='MQ4 (ppm)', color='green')
ax.set_title("Overall MQ2 & MQ4 Concentrations (ppm)\n(Readings every 30 minutes)")
ax.set_xlabel("Time")
ax.set_ylabel("Concentration (ppm)")
ax.legend()
ax.grid(True)
plt.tight_layout(rect=[0, 0.3, 1, 1])

plt.figtext(
    0.05, 0.02,
    explanation_text,
    wrap=True,
    ha="left",
    va="bottom",
    fontsize=11,
    bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=0.8")
)

plt.savefig(os.path.join(save_folder, "Overall_MQ2_MQ4_ppm.png"), dpi=300)
plt.close()

print("✅ Graphs generated successfully (ppm)")


✅ Graphs generated successfully (ppm)


In [23]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from datetime import date, timedelta

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"
os.makedirs(save_folder, exist_ok=True)

# --- Load dataset ---
df = pd.read_csv(file_path)
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime (your format) ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M', errors='coerce')

# --- Add day and half-hour index ---
df['Day'] = df['Time'].dt.date
df['HalfHourIndex'] = df.groupby('Day').cumcount()

# --- Generate artificial day sequence starting from 2025-10-03 ---
unique_days = df['Day'].unique()
base_date = date(2025, 10, 3)
display_days = [base_date + timedelta(days=i) for i in range(len(unique_days))]

# --- Readable, well-spaced explanation text ---
explanation_text = (
    "Y-axis shows gas concentration in ppm (parts per million). Higher = more target gas.\n\n"
    "Each data point is derived from ADC readings:\n"
    "   ADC → Vout → Rs = RL × (5 − Vout) / Vout → (Rs/R0) → ppm conversion.\n\n"
    "Rs = sensor resistance (changes with gas).\n"
    "R0 = resistance in clean air. RL = load resistor.\n\n"
    "Rs/R0 reflects the gas exposure ratio — lower Rs/R0 → higher gas concentration.\n\n"
    "• MQ-2 ppm ≈ 50 × (Rs/R0)^−2.3  → detects smoke, LPG, propane, etc.\n"
    "• MQ-4 ppm ≈ 1000 × (Rs/R0)^−2.95 → detects methane (CH₄) concentration.\n\n"
    "As gas concentration increases, Rs/R0 decreases → ppm increases.\n"
    "Peaks = higher gas presence, flat regions = clean air.\n\n"
    "Start date: Only includes data from 2025-10-03 onward."
)

# --- Function to create individual plots ---
def make_graph(subset, sensor_name, color, y_values, filename, display_day, i):
    fig, ax = plt.subplots(figsize=(12, 10))
    plt.subplots_adjust(bottom=0.3)  # more bottom space for large text

    ax.plot(subset['HalfHourIndex'], y_values, marker='o', color=color)
    ax.set_title(f"{sensor_name} (ppm) - Day {i} ({display_day})\n(Readings every 30 minutes)")
    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel(f"{sensor_name} Concentration (ppm)")

    hour_ticks = np.arange(0, 48, 2)
    minor_ticks = np.arange(1, 48, 2)
    hour_labels = np.arange(0, 24, 1)

    ax.set_xticks(hour_ticks)
    ax.set_xticklabels(hour_labels)
    ax.tick_params(axis='x', which='major', length=8, width=1.2)
    ax.tick_params(axis='x', which='minor', length=4, color='gray')
    ax.set_xticks(minor_ticks, minor=True)
    ax.grid(True, which='major', linestyle='-', linewidth=0.6)
    ax.grid(True, which='minor', linestyle=':', linewidth=0.4)

    # Explanation text block (large, spaced)
    plt.figtext(
        0.05, 0.02,
        explanation_text,
        wrap=True,
        ha="left",
        va="bottom",
        fontsize=12,
        linespacing=1.5,
        bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=1.0")
    )

    plt.savefig(os.path.join(save_folder, filename), dpi=300)
    plt.close()


# --- Create graphs for each day ---
for i, (real_day, display_day) in enumerate(zip(unique_days, display_days), start=1):
    subset = df[df['Day'] == real_day].reset_index(drop=True)

    make_graph(subset, "MQ2", "blue", subset['mq2'], f"Day{i}_MQ2_ppm.png", display_day, i)
    make_graph(subset, "MQ4", "green", subset['mq4'], f"Day{i}_MQ4_ppm.png", display_day, i)

    # Combined plot
    fig, ax = plt.subplots(figsize=(12, 10))
    plt.subplots_adjust(bottom=0.3)
    ax.plot(subset['HalfHourIndex'], subset['mq2'], marker='o', label='MQ2 (ppm)', color='blue')
    ax.plot(subset['HalfHourIndex'], subset['mq4'], marker='o', label='MQ4 (ppm)', color='green')
    ax.set_title(f"MQ2 & MQ4 (ppm) - Day {i} ({display_day})\n(Readings every 30 minutes)")
    ax.set_xlabel("Time (Hours)")
    ax.set_ylabel("Concentration (ppm)")
    ax.legend()
    ax.grid(True)

    plt.figtext(
        0.05, 0.02,
        explanation_text,
        wrap=True,
        ha="left",
        va="bottom",
        fontsize=12,
        linespacing=1.5,
        bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=1.0")
    )

    plt.savefig(os.path.join(save_folder, f"Day{i}_Merged_ppm.png"), dpi=300)
    plt.close()


# --- Overall timeline plot ---
fig, ax = plt.subplots(figsize=(14, 10))
plt.subplots_adjust(bottom=0.3)
ax.plot(df['Time'], df['mq2'], label='MQ2 (ppm)', color='blue')
ax.plot(df['Time'], df['mq4'], label='MQ4 (ppm)', color='green')
ax.set_title("Overall MQ2 & MQ4 Concentrations (ppm)\n(Readings every 30 minutes)")
ax.set_xlabel("Time")
ax.set_ylabel("Concentration (ppm)")
ax.legend()
ax.grid(True)
plt.tight_layout(rect=[0, 0.35, 1, 1])

plt.figtext(
    0.05, 0.02,
    explanation_text,
    wrap=True,
    ha="left",
    va="bottom",
    fontsize=12,
    linespacing=1.5,
    bbox=dict(facecolor='whitesmoke', alpha=0.9, boxstyle="round,pad=1.0")
)

plt.savefig(os.path.join(save_folder, "Overall_MQ2_MQ4_ppm.png"), dpi=300)
plt.close()

print("✅ Graphs generated with expanded, readable explanation text block!")


✅ Graphs generated with expanded, readable explanation text block!


In [25]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import os

# --- File paths ---
file_path = r"D:\ChipVista\Projects\decaying fruits\DATALOG.csv"
save_folder = r"D:\ChipVista\Projects\decaying fruits"
os.makedirs(save_folder, exist_ok=True)

# --- Load dataset ---
df = pd.read_csv(file_path)
df = df[['mq2', 'mq4', 'Time']]

# --- Convert Time column to datetime ---
df['Time'] = pd.to_datetime(df['Time'], format='%d/%m/%Y %H:%M', errors='coerce')

# --- MQ2 overall plot ---
fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(df['Time'], df['mq2'], label='MQ2 (ppm)', color='blue')
ax.set_title("Overall MQ2 Concentration (ppm)\n(Readings every 30 minutes)")
ax.set_xlabel("Time")
ax.set_ylabel("Concentration (ppm)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend()

# ✅ Force full date (year + month + day + hour)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(save_folder, "Overall_MQ2_ppm.png"), dpi=300)
plt.close()

# --- MQ4 overall plot ---
fig, ax = plt.subplots(figsize=(14, 8))
ax.plot(df['Time'], df['mq4'], label='MQ4 (ppm)', color='green')
ax.set_title("Overall MQ4 Concentration (ppm)\n(Readings every 30 minutes)")
ax.set_xlabel("Time")
ax.set_ylabel("Concentration (ppm)")
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend()

# ✅ Same here for full date formatting
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d %H:%M'))
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig(os.path.join(save_folder, "Overall_MQ4_ppm.png"), dpi=300)
plt.close()

print("✅ graph generated with full year on x-axis")


✅ graph generated with full year on x-axis
